In [1]:
! pip install -q transformers
! pip install -q bitsandbytes --upgrade
! pip install -q accelerate
! pip install -q PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 27.4 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

In [2]:
import os
import fitz
import pandas as pd
import numpy as np
import shutil
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import login
from accelerate import Accelerator, dispatch_model

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, BitsAndBytesConfig, AutoModelForCausalLM


from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [4]:
secret_value_0 = user_secrets.get_secret("cure_bench_1")
login(secret_value_0)

## Now download the model

In [7]:
model_path = "/kaggle/input/qwen3-4b-finetuned-2-for-university-handbook" # "Qwen/Qwen3-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_compute_dtype= torch.bfloat16,
    bnb_4bit_use_double_quant= True,
)

model =  AutoModelForCausalLM.from_pretrained( # AutoModelForSequenceClassification.from_pretrained
    model_path,  
    # num_labels = 2, # for "AutoModelForSequenceClassification" use "num_labels"
    torch_dtype="auto",
    device_map= "balanced",
    # quantization_config=bnb_config, # 2nd retraing , comment it
    trust_remote_code=True,
)

model.config.use_cache = False #Uses less memory during training, need to be set to True during inference
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()


tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True,)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

            
# device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
# model.to(device)

## for validating the model: Create some QNA

In [8]:
questions = [
    "Is there a discount for students who score 90% or higher in the General Secondary School Examination?",
    "Where can students pay their tuition fees besides the University finance department?",
    "Who determines the University tuition fees?",
    "Can tuition fees increase after a student has enrolled?",
    "What are the payment methods accepted by the University?",
    "What is the policy for paying fees in two installments?",
    "What is the policy for paying fees in three installments?",
    "Are IEP students eligible for merit-based tuition reductions?",
    "What financial assistance is available for needy students?",
    "Does the University provide financial assistance in cash?",
    "Is there a special tuition reduction for outstanding high school performance?",
    "When are students eligible for financial assistance based on GPA?",
    "What happens if a course is canceled by the University?",
    "Can the University cancel a course if insufficient students register?",
    "Do students have to pay fees for thesis completion?",
    "What is the purpose of student-run media at UOS?",
    "Can a student publish content representing UOS without permission?",
    "Are students allowed to use University logos on media?",
    "What should students avoid posting online?",
    "Can students identify their personal views online?",
    "What actions can the University take for non-compliance with media policies?",
    "What federal law applies to publishing offensive material online?",
    "Who is responsible for IT resource management at UOS?",
    "What is considered inappropriate use of University electronic resources?",
    "Can students copy licensed software?",
    "Are students allowed to download software without authorization?",
    "Can students inject viruses or hack University systems?",
    "What is the policy for social media administrators at UOS?",
    "Are students allowed to comment on regulatory or legal issues on behalf of the University?",
    "What platforms are included in UOS social media?",
    "What is the purpose of social media policy at UOS?",
    "Can employees post confidential information on official social media sites?",
    "Does UOS allow freedom of speech on social media?",
    "What is the role of the social media administrator?",
    "Can social media content posted by the public be considered a state record?",
    "What financial discount is offered to students with siblings at UOS?",
    "Can financial assistance be combined with merit-based tuition reductions?",
    "What are the criteria for receiving financial assistance for needy students?",
    "What happens if a student drops a course after the first week but before the third week?",
    "Are students allowed to use social media for personal purposes during academic hours?",
    "What happens if a student violates the code of ethics on social media?",
    "Can students share confidential files or passwords?",
    "Are students allowed to represent others online without authorization?",
    "Is the University responsible for social media content posted by students?",
    "Who decides on disciplinary sanctions for student misconduct?",
    "What is the maximum period for filing a disciplinary complaint?",
    "Can disciplinary sanctions be applied retroactively if a student withdraws?",
    "What rights do students have when participating in disciplinary proceedings?",
    "What are the students' rights at the University of Sharjah?",
    "What are the students' responsibilities at UOS?",
    "What is the maximum time frame for disciplinary proceedings to commence?",
    "How are students notified about disciplinary sessions?"
]

actualAns = [
    "Yes, a 50% reduction of tuition fees is granted for the first semester, excluding full scholarship recipients.",
    "Students may also pay fees through Sharjah Islamic Bank to save time and effort.",
    "Tuition fees are determined annually by the Board of Trustees with approval from the Supreme President of the University.",
    "Yes, fees may increase between 3% and 5% annually for all students, with higher increases typically applied to new students.",
    "Payments can be made in cash, by cheque issued to the University of Sharjah, or by credit card.",
    "The first installment is due at registration and the second by a predated check due two months later.",
    "The first installment must be 50% of the fees, with the remaining two due one week before mid-semester and final exams.",
    "No, IEP students are not eligible for merit-based reductions.",
    "Needy students may receive fee credit if registered for at least 15 credit hours and maintain a GPA of 3 or higher.",
    "No, assistance is only given as credit towards tuition fees.",
    "Yes, students with 90% or higher in the General Secondary School Examination get a 50% tuition reduction in the first semester.",
    "Students with a GPA of 3.6 or higher may qualify for additional financial support.",
    "A 100% refund of tuition fees is given if the University cancels the course.",
    "Yes, the University may cancel a course if the minimum required number of students is not met.",
    "No fees are required for thesis completion.",
    "Student-run media allows students to engage in communication activities while following University policies and protecting professional reputations.",
    "No, students must obtain permission from the media center before launching any media activities representing the University.",
    "Yes, but only with prior approval from the University.",
    "Students should avoid sharing confidential information, offensive content, and commenting on controversial subjects on behalf of the University.",
    "Yes, students must clearly identify their personal views as separate from University positions.",
    "The University, reinforced by the Chancellor, may take disciplinary action for non-compliance.",
    "Federal Decree-Law No (5) of 2012 regarding Information Technology Crimes applies to publishing offensive or defamatory material.",
    "Students are responsible for the proper care and use of IT resources under their direct control.",
    "Using resources for non-University activities, accessing objectionable material, hacking, or violating privacy is inappropriate.",
    "No, copying licensed software without authority or license is prohibited.",
    "No, downloading software without proper authorization or license is prohibited.",
    "No, injecting viruses or hacking systems is a serious offense.",
    "Administrators must be permanent employees, manage content, ensure accessibility, and assign an alternate during absence.",
    "No, students must refrain from commenting on such subjects on behalf of the University.",
    "YouTube, Instagram, Facebook, Twitter, Flickr, and Live Chat are official UOS social media platforms.",
    "The policy ensures quality, legal compliance, and professional standards for all official University social media content.",
    "No, confidential or personal information must be removed promptly if posted.",
    "Yes, within the framework of UAE bylaws and University policies.",
    "The administrator manages content, ensures accessibility, and maintains compliance with IT policies and CAA standards.",
    "Yes, content on official University social media sites is considered a state record and must follow retention requirements.",
    "A 10% discount on net tuition fees is offered to each sibling registered in the same semester.",
    "No, certain financial assistance, such as for distinguished students, cannot be combined with other financial aid.",
    "Students must be enrolled in at least 15 credit hours and have a GPA of 3 or above in the previous semester.",
    "Students receive a 100% refund, and a 'Withdrawal without Penalty' (W) grade is entered.",
    "Students must use social media responsibly and in line with University policies, especially during academic duties.",
    "The University may take disciplinary action, and violations may also be subject to federal law regarding IT crimes.",
    "No, sharing sensitive files or passwords without authorization is strictly prohibited.",
    "No, students cannot represent others unless explicitly authorized.",
    "The University monitors official accounts but does not hold responsibility for personal views if clearly identified as personal.",
    "The Disciplinary Committee, Provost, and Vice Chancellor decide on sanctions, and students may appeal to the Chancellor.",
    "Complaints should be submitted no later than two weeks after the incident, unless there is a valid justification for delay.",
    "Yes, sanctions can be applied upon re-registration if the student withdraws after proceedings have started.",
    "Students have the right to be notified, attend sessions, present opinions, seek legal advice, and appeal decisions.",
    "Students have the right to pursue education, access University facilities, receive professional and equal education, participate in clubs, submit grievances, and appeal grades.",
    "Students must adhere to University bylaws, respect Islamic ethics and UAE cultural values, maintain academic integrity, fulfill academic obligations, and provide correct personal data.",
    "Disciplinary proceedings should commence no later than one month after the incident or identification of the violator.",
    "Students are notified in writing at least five working days before the session, including date, venue, and method of notification chosen by the Dean for Student Affairs."
]

In [ ]:
for x in range(0, len(questions)): 
    messages = [
        {
            "role": "system",
            "content": "You are a Question-Answering Agent for Student, Profssors, Faculty stuffs for university of Sharjah"
        },
        {
            "role": "user",
            "content": f"Question: {questions[x]}"
        }
    ]
    encoding = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
        enable_thinking = False
    )
    output_ids = model.generate(
        **encoding,
        max_new_tokens=1000,   # how long answer can be
        do_sample=True,       # enable randomness
        temperature=0.5,       # creativity
        top_p=0.9,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
    )
    # Decode
    answer = tokenizer.decode(output_ids[0][encoding["input_ids"].shape[-1]:], skip_special_tokens=True)
    print("Model answer:", answer)
    print("Question:", questions[x])
    print("Actual answer: ", actualAns[x])
    print()
    print()
    print()

In [11]:
messages = [
    {
        "role": "system",
        "content": "You are a Question-Answering Agent for Student, Profssors, Faculty stuffs for university of Sharjah"
    },
    {
        "role": "user",
        "content": f"Question: what are the housing opportunities at UOS?"
    }
]
encoding = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking = True
)
output_ids = model.generate(
    **encoding,
    max_new_tokens=2000,   # how long answer can be
    do_sample=True,       # enable randomness
    temperature=0.5,       # creativity
    top_p=0.9,
    # repetition_penalty=1.2,
    eos_token_id=tokenizer.eos_token_id,
)
# Decode
answer = tokenizer.decode(output_ids[0][encoding["input_ids"].shape[-1]:], skip_special_tokens=True)
print("Model answer:", answer)
print()
print()
print()

Model answer: <think>
Okay, the user is asking about housing opportunities at UOS. I need to provide a comprehensive answer. Let me start by recalling what I know about UOS. UOS is the University of Sharjah, located in the UAE. Housing options for students are usually a key concern. I should break down the different types of housing available.

First, there's the on-campus housing. I remember that UOS has dormitories. I should mention the types of dorms, like single, double, or family rooms. Also, the facilities like laundry, common areas, and maybe the location relative to campus. I should check if there are any specific details like the number of rooms or availability for international students. Wait, I think UOS offers both male and female dorms. Also, maybe the dorms are for both local and international students. I should mention that.

Then there's off-campus housing. International students might prefer this. I need to include options like private rentals, shared apartments, or ho